# 02 · Split the SMP "usual" into train / eval / test

This notebook now does two main tasks:
* Create train / eval / test splits for the SMP2020-EWECT usual subset
* Build the matched slang vs. non-slang datasets needed for RQ1: does slang cause more sentiment loss in translation from Chinese to English?


### Rules for the split
1. Only the **usual** subset
2. Keep identical text in the same split so duplicate posts cannot leak across splits
3. Stratify by **emotion label × slang/no-slang**.
4. Route HADQAET-overlapping texts away from SMP eval/test
> **Important:** putting HADQAET-overlapping posts in the SMP `train` partition here only keeps them out of the SMP eval/test experiment sets.  
> Notebook 04 must still **exclude `in_hadqaet == True` rows from mBERT training** if HADQAET will be used as a human validation benchmark.

### RQ1 matched-control design
For every **unique slang-containing post** in eval/test, choose one non-slang control that:
- comes from the same split
- has the same gold emotion label
- has the closest possible Chinese character length
- is not reused for another slang post


### Outputs
- `data/processed/usual_splits.csv`
- `data/processed/rq1_matched_eval.csv`
- `data/processed/rq1_matched_test.csv`
- `data/processed/rq1_matched_controls.csv`
- `data/processed/slang_match_audit_sample.csv`

## 0. Setup

In [27]:
import os
import random
import pandas as pd
import numpy as np
from scipy.optimize import linear_sum_assignment

In [2]:
# CAN CHANGE THESE
eval_fraction = 0.10
test_fraction = 0.10 # everything else (~80%) goes to train
random_seed = 266

# Folders (same logic as notebook 01)
current_folder = os.getcwd()
if os.path.basename(current_folder) == "notebooks":
    project_root = os.path.dirname(current_folder)
else:
    project_root = current_folder

processed_folder = os.path.join(project_root, "data", "processed")
print("Saving to:", processed_folder) #including this because I messed up the save location before

Saving to: /Users/kathuynh/Documents/Berkeley/DataSCI266/W266-slang-emotion-mt/data/processed


## 1. Load the data

In [3]:
smp = pd.read_csv(os.path.join(processed_folder, "smp2020_ewect.csv"))
hadqaet = pd.read_csv(os.path.join(processed_folder, "hadqaet_annotated.csv"))
cheer = pd.read_csv(os.path.join(processed_folder, "cheer_lexicon.csv"))

# Keep only the 'usual' subset.
usual = smp[smp["subset"] == "usual"].copy()

# Rename SMP's original split column so it doesn't clash with our new one (not using their defined split because of the HADQAET overlap)
usual = usual.rename(columns={"split": "orig_split"})
print("Usual posts:", len(usual))
print("HADQAET rows:", len(hadqaet))
print("CHEER buzzwords:", len(cheer))

Usual posts: 34768
HADQAET rows: 5538
CHEER buzzwords: 1127


In [4]:
# Remove empty posts
keep_rows = []
for text in usual["text"]:
    # pd.isna gets the cells pandas read as missing and strip() == "" gets the blank ones
    if pd.isna(text) or str(text).strip() == "":
        keep_rows.append(False)
    else:
        keep_rows.append(True)

rows_before = len(usual)
usual = usual[keep_rows].copy() #using .copy() because I always get messed up when making copies of data with python's reference system
print("Removed: ", rows_before - len(usual))
print(len(usual), " posts left")

Removed:  2
34766  posts left


## 2. Mark HADQAET posts

Notebook 01 saved the matching SMP id for each HADQAET row in the `smp_post_id` column. We:
- collect those ids
- mark every usual post whose text matches one of those post (also catches repeated copies of the same text)

In [6]:
# The set of SMP post ids that HADQAET matched
hadqaet_post_ids = set()
for post_id in hadqaet["smp_post_id"]:
    if pd.isna(post_id):
        continue
    hadqaet_post_ids.add(post_id)

# The texts of those posts
hadqaet_texts = set()
for index, row in usual.iterrows():
    if row["post_id"] in hadqaet_post_ids:
        hadqaet_texts.add(row["text"].strip()) #strip the text for a better comparison

# mark every post whose text is one of those texts
in_hadqaet = []
for text in usual["text"]:
    cleaned_text = text.strip()# remove spaces
    if cleaned_text in hadqaet_texts:# check if this text one of the HADQAET texts
        in_hadqaet.append(True)
    else:
        in_hadqaet.append(False)

usual["normalized_text"] = usual["text"].astype(str).str.strip()
usual["in_hadqaet"] = in_hadqaet #add to text table

print(f"HADQAET post ids found in usual: {len(hadqaet_post_ids)}")
print(f"Rows marked as HADQAET (including repeats): {sum(in_hadqaet)}" )

HADQAET post ids found in usual: 5510
Rows marked as HADQAET (including repeats): 5977


## 3. Mark posts that (probably) contain slang

The current project definition of a slang-containing post is:

> the post contains at least one CHEER buzzword as an exact character substring.

This is a deliberately simple dictionary-based detector. We check longer buzzwords first and save all matched expressions, not just a Boolean flag.

We also create a deterministic **incorrect-definition lookup** for RQ3*. Each CHEER term is paired with the definition of a different CHEER term using a fixed seed, so the wrong-context condition is reproducible

*does incorrect slang definition increase sentiment loss in translation?

In [11]:
# Clean and dedup CHEER and make one row per buzzword
cheer = cheer.dropna(subset=["buzzword", "definition"]).copy()
cheer["buzzword"] = cheer["buzzword"].astype(str)
cheer["definition"] = cheer["definition"].astype(str)
cheer = cheer.drop_duplicates(subset=["buzzword"], keep="first").reset_index(drop=True)

definition_lookup = dict(zip(cheer["buzzword"], cheer["definition"]))

# Sort by length from longest so it's easy to inspect when one expression contains another
buzzword_lookup = sorted(definition_lookup.keys(), key=lambda x: (-len(x), x))

# ----------------------------------------------- #
# Create a deterministic wrong-definition mapping
rng = np.random.default_rng(random_seed)
shuffled_terms = buzzword_lookup.copy()
rng.shuffle(shuffled_terms)

wrong_definition_lookup = {}
for i, term in enumerate(shuffled_terms):
    wrong_term = shuffled_terms[(i + 1) % len(shuffled_terms)] # rotate by one so no term accidentally receives its correct def
    wrong_definition_lookup[term] = definition_lookup[wrong_term]

# ----------------------------------------------- #
def find_buzzwords(text):
    text = str(text).strip()
    return [word for word in buzzword_lookup if word in text]

found_lists = usual["normalized_text"].map(find_buzzwords)

usual["has_slang_simple"] = found_lists.map(bool)
usual["slang_terms"] = found_lists.map(lambda xs: " || ".join(xs))
usual["slang_definitions"] = found_lists.map(
    lambda xs: " || ".join(f"{x}: {definition_lookup[x]}" for x in xs)
)
usual["incorrect_slang_definitions"] = found_lists.map(
    lambda xs: " || ".join(f"{x}: {wrong_definition_lookup[x]}" for x in xs)
)

print("Posts with a CHEER buzzword:", usual["has_slang_simple"].sum(), "of", len(usual))
print("Unique CHEER terms matched:", len({w for xs in found_lists for w in xs}))

Posts with a CHEER buzzword: 885 of 34766
Unique CHEER terms matched: 156


### Save a slang-match audit sample

Substring matching can produce false positives when a CHEER expression is used in a normal/non-slang sense 
This sample is for manual inspection 

In [12]:
slang_rows_for_audit = usual[usual["has_slang_simple"]].copy()
audit_n = min(100, len(slang_rows_for_audit))

audit = slang_rows_for_audit.sample(audit_n, random_state=random_seed)[
    ["post_id", "label", "text", "slang_terms", "slang_definitions"]
].copy()

audit["manual_is_true_slang"] = ""
audit["manual_notes"] = ""

audit_path = os.path.join(processed_folder, "slang_match_audit_sample.csv")
audit.to_csv(audit_path, index=False, encoding="utf-8-sig")

print("Saved audit sample:", audit_path)
audit.head()

Saved audit sample: /Users/kathuynh/Documents/Berkeley/DataSCI266/W266-slang-emotion-mt/data/processed/slang_match_audit_sample.csv


,post_id,label,text,slang_terms,slang_definitions,manual_is_true_slang,manual_notes
19463,usual_train_19464,angry,谁说的人生如梦，然而作了几晚梦评估竟tmd还没完，明早还要静静地看狗腿砸们装逼起飞，坏了我的...,装逼,装逼: “装逼”一词源于中文网络，本指“装作很厉害”的行为，类似“打肿脸充胖子”，带有贬义。...,,
12001,usual_train_12002,angry,道理都懂，但该怨的还是会怨，该骂的还是会骂，该哭的也还是会哭，毕竟心里的难受不是道理所能释怀...,理都懂,理都懂: “理都懂”是“道理我都懂”的缩略形式，常用于表达明知某个道理，却依然无法改变现状的...,,
17046,usual_train_17047,surprise,原来bisu在法语的意思是………………………………………………………………………………………...,么么哒,么么哒: “么么哒”是一个网络流行词，是“么么”的叠词，表示亲吻的声音，通常用于表达爱意、亲...,,
23966,usual_train_23967,sad,明天又是通宵大夜班，我想静静奥，大夜后双技能💣💣💣,我想静静,"我想静静: ""我想静静""是一句俗语和网络流行语，意为""让我静一静""，表达了遭遇不快之事时寻求...",,
12676,usual_train_12677,fear,乱入一只猫太凶残了太可怕最后那个最吓人了！太吓人了,乱入,乱入: 原意指未经允许闯入，现多用于ACG领域，指不同作品中的人物、道具、场景等出现在一起的...,,


## 4. Group identical text

Each unique text becomes one "group" that carries all of its rows with it (since some text is repeated).

Repeated text can have two different labels (maybe annotators disagreed). For balancing purposes (maintaing the emotion split percentages) we give each group the label of its first row, (but the origianl labels stay unchanged in the saved file)

In [13]:
groups = {} # text -> information about that text

for index, row in usual.iterrows():
    text = row["text"].strip()

    if text not in groups: # First time we see this text: create its group
        groups[text] = {
            "label": row["label"],
            "has_slang": row["has_slang_simple"],
            "in_hadqaet": row["in_hadqaet"],
            "row_count": 0,
        }

    groups[text]["row_count"] = groups[text]["row_count"] + 1

print("Rows:", len(usual))
print("Unique texts (groups):", len(groups))

Rows: 34766
Unique texts (groups): 33605


## 5. Do the tain/eval/test splits

Splitting based on label (emotion) and slang/no-slang (combination of label + slang/no-slang, e.g. "angry + slang"). I googled it and these are called "strata":

1. Count its unique texts and work out how many should go to eval and test (10% each).
2. Send every HADQAET text in that split to train.
3. Shuffle the remaining texts and fill eval, then test. Whatever is left goes to train.

Because HADQAET texts can only go to train, eval and test are filled entirely from non-HADQAET texts.

In [14]:
# A "stratum" is a group of texts that have the same label AND contain/don't contain slang,
# for example "angry + slang" or "happy + no slang".
strata = {} #stratum name -> list of texts in that stratum

for text in groups:
    # Look up this text's label (ex "angry")
    label = groups[text]["label"]

    # Turn True/False into a readable word
    if groups[text]["has_slang"]:
        slang_part = "slang"
    else:
        slang_part = "no slang"

    # Build the stratum name, e.g. "angry + slang"
    stratum_name = label + " + " + slang_part

    # If this is the first text in this stratum, start an empty list for it
    if stratum_name not in strata:
        strata[stratum_name] = []

    # Add the text to its stratum's list
    strata[stratum_name].append(text)

# Show how many texts ended up in each stratum
for stratum_name in sorted(strata):
    print(stratum_name.ljust(22), len(strata[stratum_name]), "texts")

angry + no slang       9211 texts
angry + slang          346 texts
fear + no slang        1391 texts
fear + slang           55 texts
happy + no slang       6559 texts
happy + slang          183 texts
neutral + no slang     7085 texts
neutral + slang        44 texts
sad + no slang         6052 texts
sad + slang            141 texts
surprise + no slang    2482 texts
surprise + slang       56 texts


In [15]:
random.seed(random_seed)

split_of_text = {} #text -> "train", "eval", or "test"

# Go through the strata in alphabetical order
stratum_names_in_order = sorted(strata)

for stratum_name in stratum_names_in_order:
    # Sort the texts first so starting order is always the same
    texts_in_stratum = sorted(strata[stratum_name])

    # How many texts should go to eval and to test?
    total_count = len(texts_in_stratum)
    eval_count = round(total_count * eval_fraction)
    test_count = round(total_count * test_fraction)

    # Separate HADQAET texts from all other texts
    hadqaet_texts_here = []
    other_texts_here = []
    for text in texts_in_stratum:
        if groups[text]["in_hadqaet"] == True:
            hadqaet_texts_here.append(text)
        else:
            other_texts_here.append(text)

    # Shuffle the non-HADQAET texts
    random.shuffle(other_texts_here)

    # cut the shuffled list into three pieces
    # Example with eval_count = 3 and test_count = 3:
    #   positions 0, 1, 2   -> eval
    #   positions 3, 4, 5   -> test
    #   positions 6 onwards -> train
    eval_end = eval_count # eval stops here
    test_end = eval_count + test_count # test stops here
    eval_texts = other_texts_here[0:eval_end]
    test_texts = other_texts_here[eval_end:test_end]
    leftover_texts = other_texts_here[test_end:] # everything after test, will go into train

    # record the split for every text
    for text in eval_texts:
        split_of_text[text] = "eval"

    for text in test_texts:
        split_of_text[text] = "test"

    for text in leftover_texts:
        split_of_text[text] = "train"

    for text in hadqaet_texts_here:
        split_of_text[text] = "train"# HADQAET texts always go to train

    # Print a summary line for this stratum
    print(f"{stratum_name}: Total ({total_count}), Eval ({len(eval_texts)}), Test ({len(test_texts)}), Train ({len(leftover_texts) + len(hadqaet_texts_here)})")

angry + no slang: Total (9211), Eval (921), Test (921), Train (7369)
angry + slang: Total (346), Eval (35), Test (35), Train (276)
fear + no slang: Total (1391), Eval (139), Test (139), Train (1113)
fear + slang: Total (55), Eval (6), Test (6), Train (43)
happy + no slang: Total (6559), Eval (656), Test (656), Train (5247)
happy + slang: Total (183), Eval (18), Test (18), Train (147)
neutral + no slang: Total (7085), Eval (708), Test (708), Train (5669)
neutral + slang: Total (44), Eval (4), Test (4), Train (36)
sad + no slang: Total (6052), Eval (605), Test (605), Train (4842)
sad + slang: Total (141), Eval (14), Test (14), Train (113)
surprise + no slang: Total (2482), Eval (248), Test (248), Train (1986)
surprise + slang: Total (56), Eval (6), Test (6), Train (44)


In [16]:
# Copy each text's split back onto every row with that text
new_split = []
for text in usual["normalized_text"]:
    new_split.append(split_of_text[text.strip()])
usual["split"] = new_split

In [17]:
usual["mbert_train_eligible"] = (usual["split"] == "train") & (~usual["in_hadqaet"])
assert usual["split"].notna().all()

## 6. Check the split

In [18]:
# Check size of each split

n_train = (usual["split"] == "train").sum()
n_eval = (usual["split"] == "eval").sum()
n_test = (usual["split"] == "test").sum()

print(f"Train: {n_train} rows, {round(n_train / len(usual) * 100, 1)}%")
print(f"Eval: {n_eval} rows, {round(n_eval / len(usual) * 100, 1)}%")
print(f"Test: {n_test} rows, {round(n_test / len(usual) * 100, 1)}%")

Train: 27864 rows, 80.1%
Eval: 3457 rows, 9.9%
Test: 3445 rows, 9.9%


In [19]:
# Check label mix per split (each column should look about the same)

labels = ["angry", "happy", "sad", "fear", "surprise", "neutral"]

print("label      train %   eval %   test %")
for label in labels:
    line = label
    for split in ["train", "eval", "test"]:
        rows_in_split = usual[usual["split"] == split]
        percent = (rows_in_split["label"] == label).sum() / len(rows_in_split) * 100
        line = f"{line}     {round(percent, 1)}"
    print(line) #sorry for ugly print

label      train %   eval %   test %
angry     30.1     29.7     29.5
happy     19.5     19.5     19.6
sad     17.9     18.0     18.0
fear     4.4     4.4     4.4
surprise     7.5     7.7     7.6
neutral     20.5     20.7     20.9


In [20]:
# Check slang share per split

for split in ["train", "eval", "test"]:
    rows_in_split = usual[usual["split"] == split]
    n_slang = rows_in_split["has_slang_simple"].sum()
    percent = n_slang / len(rows_in_split) * 100
    print(f"{split}: slang posts: {n_slang} ({round(percent, 2)}%)")

train: slang posts: 702 (2.52%)
eval: slang posts: 94 (2.72%)
test: slang posts: 89 (2.58%)


In [21]:
# Check the two most important rules for data leakage

# no HADQAET post in eval or test
hadqaet_rows = usual[usual["in_hadqaet"] == True]
hadqaet_outside_train = (hadqaet_rows["split"] != "train").sum()
print("HADQAET rows outside train (must be 0):", hadqaet_outside_train)

# no text appears in more than one split
splits_per_text = {}
for index, row in usual.iterrows():
    text = row["text"].strip()
    if text not in splits_per_text:
        splits_per_text[text] = set()
    splits_per_text[text].add(row["split"])

texts_in_several_splits = 0
for text in splits_per_text:
    if len(splits_per_text[text]) > 1:
        texts_in_several_splits = texts_in_several_splits + 1
print("Texts found in more than one split (must be 0):", texts_in_several_splits)


HADQAET rows outside train (must be 0): 0
Texts found in more than one split (must be 0): 0


## 7. Build matched non-slang controls for RQ1*
The raw slang and non-slang groups have different emotion distributions. A direct overall comparison could mistake an emotion effect for slang effect so we need to construct matched pairs in eval and test datasets (since train set is used for training purposes only we do not need to construct matched pairs)
> What the pair looks like: one slang post <-> one non-slang post with the same emotion label and similar length

Note: We use unique text only, so a duplicated Weibo post is not counted as multiple independent observations.

*RQ1: does slang cause more sentiment loss in translation?

In [24]:
# One representative row per unique text
# Without drop_duplicates, the number of posts containing slang will match the above 
# eval: slang posts: 94, test: slang posts: 89
# With drop_duplicates, each split only has 83 posts containing slang
experiment_pool = (
    usual[usual["split"].isin(["eval", "test"])]
    .drop_duplicates(subset=["normalized_text"], keep="first")
    .copy()
)


experiment_pool["char_len"] = experiment_pool["normalized_text"].str.len()

print("Unique eval/test posts:", len(experiment_pool))
print()
print("Unique slang posts by split:")
print(
    experiment_pool[experiment_pool["has_slang_simple"]]
    .groupby("split")
    .size()
)



Unique eval/test posts: 6720

Unique slang posts by split:
split
eval    83
test    83
dtype: int64


In [28]:
# Within each split and emotion label, build a cost matrix:
# cost(i,j) = |length(slang_i) - length(control_j)|
# Then use minimum-cost one-to-one assignment. Each control cannot be used twice

def match_one_split(split_df, split_name):
    split_df = split_df.copy()
    pair_rows = []
    pair_counter = 0

    slang_all = split_df[split_df["has_slang_simple"]].copy()
    control_all = split_df[~split_df["has_slang_simple"]].copy()

    for label in sorted(slang_all["label"].unique()):
        slang = slang_all[slang_all["label"] == label].reset_index(drop=True)
        controls = control_all[control_all["label"] == label].reset_index(drop=True)

        cost = np.abs(
            slang["char_len"].to_numpy()[:, None]
            - controls["char_len"].to_numpy()[None, :]
        )
        
        slang_idx, control_idx = linear_sum_assignment(cost)
        
        for si, ci in zip(slang_idx, control_idx):
            srow = slang.iloc[si]
            crow = controls.iloc[ci]

            pair_counter += 1
            pair_id = f"{split_name}_{label}_{pair_counter:04d}"
            length_diff = int(abs(srow["char_len"] - crow["char_len"]))

            common = {
                "pair_id": pair_id,
                "split": split_name,
                "label": label,
                "length_diff": length_diff,
            }

            pair_rows.append({
                **common,
                "role": "slang",
                "post_id": srow["post_id"],
                "text": srow["text"],
                "char_len": int(srow["char_len"]),
                "has_slang_simple": True,
                "in_hadqaet": bool(srow["in_hadqaet"]),
                "slang_terms": srow["slang_terms"],
                "slang_definitions": srow["slang_definitions"],
                "incorrect_slang_definitions": srow["incorrect_slang_definitions"],
                "matched_post_id": crow["post_id"],
                "orig_split": srow["orig_split"],
                "orig_id": srow["orig_id"],
            })

            pair_rows.append({
                **common,
                "role": "control",
                "post_id": crow["post_id"],
                "text": crow["text"],
                "char_len": int(crow["char_len"]),
                "has_slang_simple": False,
                "in_hadqaet": bool(crow["in_hadqaet"]),
                "slang_terms": "",
                "slang_definitions": "",
                "incorrect_slang_definitions": "",
                "matched_post_id": srow["post_id"],
                "orig_split": crow["orig_split"],
                "orig_id": crow["orig_id"],
            })

    return pd.DataFrame(pair_rows)


matched_by_split = {}

for split_name in ["eval", "test"]:
    split_df = experiment_pool[experiment_pool["split"] == split_name]
    matched = match_one_split(split_df, split_name)
    matched_by_split[split_name] = matched

    print("\n", split_name.upper())
    print("Matched pairs:", matched["pair_id"].nunique())
    print("Rows:", len(matched))
    print("Pairs by emotion:")
    print(
        matched[matched["role"] == "slang"]["label"]
        .value_counts()
        .sort_index()
    )     


 EVAL
Matched pairs: 83
Rows: 166
Pairs by emotion:
label
angry       35
fear         6
happy       18
neutral      4
sad         14
surprise     6
Name: count, dtype: int64

 TEST
Matched pairs: 83
Rows: 166
Pairs by emotion:
label
angry       35
fear         6
happy       18
neutral      4
sad         14
surprise     6
Name: count, dtype: int64


### Check the matched controls

In [34]:
for split_name, matched in matched_by_split.items():
    slang = matched[matched["role"] == "slang"]
    controls = matched[matched["role"] == "control"]

    # Assert that each pair has exactly one slang row and one control row
    assert matched.groupby("pair_id").size().eq(2).all()
    assert matched.groupby("pair_id")["role"].nunique().eq(2).all()

    # Assert that posts are not reused
    assert slang["post_id"].is_unique
    assert controls["post_id"].is_unique

    # Assert that members share the same gold emotion
    assert matched.groupby("pair_id")["label"].nunique().eq(1).all()

    # Assert that control rows contain no CHEER match
    assert (~controls["has_slang_simple"].astype(bool)).all()

    # Assert that eval/test were built to exclude HADQAET overlap
    assert not matched["in_hadqaet"].astype(bool).any()

### Check balance
The slang and matched non-slang controls should have the same emotion counts.
The length differences between each pair should also be small enough 

In [33]:
for split_name, matched in matched_by_split.items():
    print("\n###", split_name)
    print(pd.crosstab(matched["label"], matched["role"]))

    display(
        matched.groupby("role")["char_len"]
        .agg(["count", "mean", "median", "min", "max"])
    )

    pair_level = (
        matched.groupby("pair_id")
        .agg(
            label=("label", "first"),
            length_diff=("length_diff", "first")
        )
        .reset_index()
    )

    display(
        pair_level.groupby("label")["length_diff"]
        .agg(["count", "mean", "median", "max"])
    )


### eval
role      control  slang
label                   
angry          35     35
fear            6      6
happy          18     18
neutral         4      4
sad            14     14
surprise        6      6


,count,mean,median,min,max
role,,,,,
control,83,54.060241,44.0,15,140
slang,83,54.072289,44.0,15,140


,count,mean,median,max
label,,,,
angry,35,0.142857,0.0,1
fear,6,0.333333,0.0,1
happy,18,0.055556,0.0,1
neutral,4,0.250000,0.0,1
sad,14,0.071429,0.0,1
surprise,6,0.166667,0.0,1



### test
role      control  slang
label                   
angry          35     35
fear            6      6
happy          18     18
neutral         4      4
sad            14     14
surprise        6      6


,count,mean,median,min,max
role,,,,,
control,83,47.590361,38.0,15,141
slang,83,47.686747,38.0,15,148


,count,mean,median,max
label,,,,
angry,35,0.200000,0.0,7
fear,6,0.166667,0.0,1
happy,18,0.055556,0.0,1
neutral,4,0.000000,0.0,0
sad,14,0.000000,0.0,0
surprise,6,0.166667,0.0,1


## 8. Save

In [35]:
# usual_splits.csv
columns_to_save = ["post_id", "split", "text", "label", "in_hadqaet", "mbert_train_eligible", "has_slang_simple", 
                   "slang_terms", "slang_definitions", "incorrect_slang_definitions", "orig_split", "orig_id"]
output_path = os.path.join(processed_folder, "usual_splits.csv")
usual[columns_to_save].to_csv(output_path, index=False, encoding="utf-8-sig")

print("Saved", output_path)
usual[columns_to_save].head()

Saved /Users/kathuynh/Documents/Berkeley/DataSCI266/W266-slang-emotion-mt/data/processed/usual_splits.csv


,post_id,split,text,label,in_hadqaet,mbert_train_eligible,has_slang_simple,slang_terms,slang_definitions,incorrect_slang_definitions,orig_split,orig_id
0,usual_train_00001,train,气死姐姐了，快二是阵亡了吗，尼玛，一个半小时过去了也没上车,angry,False,True,False,,,,train,1
1,usual_train_00002,train,妞妞啊，今天又承办了一个发文登记文号是126~嘻~么么哒~晚安哟,happy,False,True,True,么么哒,么么哒: “么么哒”是一个网络流行词，是“么么”的叠词，表示亲吻的声音，通常用于表达爱意、亲...,么么哒: 表示非常看好一对CP，希望他们马上在一起。,train,2
2,usual_train_00003,train,这里还值得注意另一个事实，就是张鞠存原有一个东溪草堂为其读书处。,neutral,False,True,False,,,,train,3
3,usual_train_00004,train,这在前华约国家(尤其是东德)使用R-73的首次联合演习期间，被一些北约组织的飞行员所证实。,neutral,False,True,False,,,,train,4
4,usual_train_00005,train,TinyThief上wii了？！,surprise,False,True,False,,,,train,5


In [36]:
all_matched = []

for split_name, matched in matched_by_split.items():
    path = os.path.join(processed_folder, f"rq1_matched_{split_name}.csv")
    matched.to_csv(path, index=False, encoding="utf-8-sig")
    print("Saved", path)
    all_matched.append(matched)

combined_matched = pd.concat(all_matched, ignore_index=True)
combined_path = os.path.join(processed_folder, "rq1_matched_controls.csv")
combined_matched.to_csv(combined_path, index=False, encoding="utf-8-sig")
print("Saved", combined_path)

print()
print("Final matched-pair counts:")
print(
    combined_matched[combined_matched["role"] == "slang"]
    .groupby("split")
    .size()
)

Saved /Users/kathuynh/Documents/Berkeley/DataSCI266/W266-slang-emotion-mt/data/processed/rq1_matched_eval.csv
Saved /Users/kathuynh/Documents/Berkeley/DataSCI266/W266-slang-emotion-mt/data/processed/rq1_matched_test.csv
Saved /Users/kathuynh/Documents/Berkeley/DataSCI266/W266-slang-emotion-mt/data/processed/rq1_matched_controls.csv

Final matched-pair counts:
split
eval    83
test    83
dtype: int64
